# Exploratory Data Analysis

What you will learn in this notebook:

| #   | Topic                     | Key Skills                                          |
| --- | ------------------------- | --------------------------------------------------- |
| 1   | Loading & inspecting data | `pd.read_csv`, `.shape`, `.head()`, `.dtypes`       |
| 2   | Summary statistics        | `.describe()`                                       |
| 3   | Missing values            | `.isnull()`, a bar chart of missing shares          |
| 4   | Table relationships       | understanding relational data, foreign keys         |
| 5   | Univariate analysis       | count with `.value_counts()`, then plot the counts  |
| 6   | Merging tables            | `pd.merge()`, enriching data                        |
| 7   | Categorical analysis      | value counts, proportions, bar charts               |
| 8   | Bivariate analysis        | grouped aggregations, a box plot                    |
| 9   | Time-based patterns       | day-of-week and hour-of-day analysis, a heatmap     |
| 10  | Correlation               | `.corr()`, a correlation heatmap, a scatter plot    |

**Dataset:** The Instacart Online Grocery Shopping Dataset 2017, read from a [public mirror on GitHub](https://github.com/subwaymatch/instacart-dataset-2017). It contains over 3 million grocery orders from more than 200,000 users. Instacart's original download page is no longer online. See the [case overview](case-overview.md) for the data dictionary and terms of use.

All visualizations use **Plotly Express** with the `simple_white` template.

## Import Libraries

Before we begin, we need to import the libraries we will use throughout this notebook.


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Use a clean, minimal chart style for all plots
px.defaults.template = "simple_white"

---

## Load and Inspect the Data

The first step in any EDA is to **load the data and get a feel for its size and structure**.

The Instacart dataset is stored as several CSV files. Let's load them all and see what we're working with.

:::{tip}

Always start by checking the **shape** (rows × columns) and the **first few rows** of each table.

:::

In [2]:
# Load all five tables
base_url = "https://raw.githubusercontent.com/subwaymatch/instacart-dataset-2017/refs/heads/main/"

orders = pd.read_csv(base_url + "orders.csv.zip", compression="zip")
order_products = pd.read_csv(base_url + "order_products_train.csv")
products = pd.read_csv(base_url + "products.csv")
aisles = pd.read_csv(base_url + "aisles.csv")
departments = pd.read_csv(base_url + "departments.csv")

Once the tables are loaded, the very first sanity check is **how big is each one?** A quick `.shape` on every DataFrame tells us the row × column count and immediately catches mistakes like an empty download or a silently-failed read.


In [3]:
# Print the shape of each table
for name, df in [
    ("orders", orders),
    ("order_products", order_products),
    ("products", products),
    ("aisles", aisles),
    ("departments", departments),
]:
    print(f"{name:20s}  →  {df.shape[0]:>10,} rows × {df.shape[1]} columns")

orders                →   3,421,083 rows × 7 columns
order_products        →   1,384,617 rows × 4 columns
products              →      49,688 rows × 4 columns
aisles                →         134 rows × 2 columns
departments           →          21 rows × 2 columns


### Peek at each table with `.head()`

`.head()` shows the first 5 rows. It is the quickest way to see what the columns contain.


In [4]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [5]:
order_products.head()

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


In [6]:
products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


### Data types with `.dtypes`

Understanding column types is critical. Numeric columns can be aggregated; categorical/object columns need special treatment.

:::{tip}

If a column you expect to be numeric shows as `object` or `str`, it may contain unexpected text or missing-value markers.

:::

In [7]:
orders.dtypes

order_id                    int64
user_id                     int64
eval_set                      str
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object

### Quick summary with `.info()`

`.info()` gives you column names, non-null counts, and data types all in one shot. Very handy!


In [8]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                str    
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), str(1)
memory usage: 198.9 MB


---

## Summary Statistics with `.describe()`

`.describe()` computes count, mean, standard deviation, min, quartiles, and max for every numeric column.

This is your first look at the **distribution** of the data: are there outliers? What is the typical range?


In [9]:
orders.describe()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01


**What to look for:**

- **count**: does it match the number of rows? If not, there are missing values.
- **min / max**: are the ranges reasonable?
- **mean vs. median (50%)**: a large gap indicates a skewed distribution.
- **std**: high standard deviation means high variability.


In [10]:
order_products.describe()

,order_id,product_id,add_to_cart_order,reordered
count,1.384617e+06,1.384617e+06,1.384617e+06,1.384617e+06
mean,1.706298e+06,2.555624e+04,8.758044e+00,5.985944e-01
std,9.897326e+05,1.412127e+04,7.423936e+00,4.901829e-01
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
25%,8.433700e+05,1.338000e+04,3.000000e+00,0.000000e+00
50%,1.701880e+06,2.529800e+04,7.000000e+00,1.000000e+00
75%,2.568023e+06,3.794000e+04,1.200000e+01,1.000000e+00
max,3.421070e+06,4.968800e+04,8.000000e+01,1.000000e+00


---

## Checking for Missing Values

Missing values can distort your analysis. Always check for them early.

We'll count missing values per column and visualize them.


In [11]:
# Count missing values in each table
for name, df in [
    ("orders", orders),
    ("order_products", order_products),
    ("products", products),
    ("aisles", aisles),
    ("departments", departments),
]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) == 0:
        print(f"✅ {name}: no missing values")
    else:
        print(f"⚠️  {name}:")
        for col, count in missing.items():
            pct = 100 * count / len(df)
            print(f"     {col}: {count:,} missing ({pct:.1f}%)")

⚠️  orders:
     days_since_prior_order: 206,209 missing (6.0%)
✅ order_products: no missing values
✅ products: no missing values
✅ aisles: no missing values
✅ departments: no missing values


:::{note} Why is `days_since_prior_order` missing?

A customer's _very first order_ has no prior order, so the field is naturally `NaN`. The 206,209 missing values match the 206,209 users in `orders` exactly: one first order per user. This is **expected**, not a data-quality issue.

:::

In [12]:
# Visualize the proportion of missing values
missing_pct = (orders.isnull().mean() * 100).reset_index()
missing_pct.columns = ["Column", "% Missing"]

fig = px.bar(
    missing_pct,
    x="Column",
    y="% Missing",
    title="Percentage of Missing Values in the Orders Table",
    text_auto=".1f",
)
fig.update_layout(yaxis_range=[0, max(missing_pct["% Missing"].max() * 1.2, 1)])
fig.show()

---

## Understanding Table Relationships

Real-world data rarely lives in a single table. The Instacart dataset has **five related tables**:

```
departments  ──┐
               ├──  products  ──┐
aisles  ───────┘                ├──  order_products  ──── orders
                                │
                          (product_id)              (order_id)
```

- **orders**: one row per order; contains user and timing info.
- **order_products**: one row per _item in an order_; links `order_id` → `product_id`.
- **products**: one row per product; links to `aisle_id` and `department_id`.
- **aisles**: lookup table for aisle names.
- **departments**: lookup table for department names.

:::{important} Key concept

A _foreign key_ (e.g., `product_id` in `order_products`) references the _primary key_ of another table (`product_id` in `products`). This is how tables are joined.

:::

In [13]:
# Verify the linking columns exist and check their cardinality
print("Unique order_ids in order_products:", order_products["order_id"].nunique())
print("Unique order_ids in orders:        ", orders["order_id"].nunique())
print()
print("Unique product_ids in order_products:", order_products["product_id"].nunique())
print("Unique product_ids in products:      ", products["product_id"].nunique())
print()
print("Unique aisle_ids in products:", products["aisle_id"].nunique())
print("Unique aisles:              ", len(aisles))
print()
print("Unique department_ids in products:", products["department_id"].nunique())
print("Unique departments:              ", len(departments))

Unique order_ids in order_products: 131209


Unique order_ids in orders:         3421083

Unique product_ids in order_products: 39123
Unique product_ids in products:       49688

Unique aisle_ids in products: 134
Unique aisles:               134

Unique department_ids in products: 21
Unique departments:               21


> **What this tells us:** `orders` lists every order for every user (3.4 million), but `order_products` covers only 131,209 of them. These are the orders in the **train** split (`eval_set == "train"`), and there is exactly one per user: that user's most recent order. So every item-level result in this notebook (products, aisles, reorders, basket sizes) comes from one recent order per customer. The [case overview](case-overview.md) explains what this means for customer-level analysis.

---

## Univariate Analysis (One Variable at a Time)

Univariate analysis examines the distribution of a **single column**. The right chart depends on the data type:

| Data Type                  | Good Charts              |
| -------------------------- | ------------------------ |
| Continuous numeric         | Histogram, box plot      |
| Discrete numeric / ordinal | Bar chart                |
| Categorical                | Bar chart (value counts) |

Let's start with some important numeric distributions.

:::{tip} Count first, then plot

`px.histogram` stores every raw value inside the figure. With 3.4 million orders that makes a notebook tens of megabytes large. The columns below are whole numbers (hours, days, positions), so we count each value with `.value_counts()` and draw the counts with `px.bar`. The chart looks the same as a histogram with one bar per value, but the figure holds only a few dozen numbers.

:::

### Distribution of `order_hour_of_day`


In [14]:
# Count orders per hour (24 numbers), then plot the counts
hour_counts = orders["order_hour_of_day"].value_counts().sort_index().reset_index()
hour_counts.columns = ["Hour of Day", "Number of Orders"]

fig = px.bar(
    hour_counts,
    x="Hour of Day",
    y="Number of Orders",
    title="When Do Customers Place Orders? (Hour of Day)",
)
fig.show()

> **Observation:** About two thirds of orders are placed between 9 AM and 5 PM. Volume is nearly flat from 10 AM to 4 PM, with 10 AM the busiest hour by a small margin. Very few orders are placed between 1 AM and 5 AM.

### Distribution of `days_since_prior_order`


In [15]:
# Count orders for each gap of 0 to 30 days (first orders have no gap and are dropped)
days_counts = (
    orders["days_since_prior_order"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .reset_index()
)
days_counts.columns = ["Days Since Prior Order", "Number of Orders"]

fig = px.bar(
    days_counts,
    x="Days Since Prior Order",
    y="Number of Orders",
    title="How Many Days Between Consecutive Orders?",
)
fig.show()

> **Observation:** The tallest bar is at **30 days**, but the column is capped at 30, so that bar means "30 days or more". The next spike is at **7 days**, which suggests weekly shoppers, with smaller bumps at 14 and 21 days. Just over half of all gaps are between 1 and 8 days.

### Distribution of `order_number` (how many orders a customer has placed)


In [16]:
# Count orders for each order sequence number (1 to 100)
order_number_counts = orders["order_number"].value_counts().sort_index().reset_index()
order_number_counts.columns = ["Order Number (nth order for user)", "Number of Orders"]

fig = px.bar(
    order_number_counts,
    x="Order Number (nth order for user)",
    y="Number of Orders",
    title="Distribution of Order Sequence Number",
)
fig.show()

> **Observation:** Order numbers 1 to 4 each have exactly 206,209 orders, one per user. The dataset only includes users with at least 4 orders, so every user reaches order 4. After that the counts fall steadily as fewer users reach each later order. No user has more than 100 orders in the dataset.

### Distribution of `add_to_cart_order`


In [17]:
# Count items for each add-to-cart position
cart_position_counts = (
    order_products["add_to_cart_order"].value_counts().sort_index().reset_index()
)
cart_position_counts.columns = ["Position Added to Cart", "Number of Items"]

fig = px.bar(
    cart_position_counts,
    x="Position Added to Cart",
    y="Number of Items",
    title="Distribution of Add-to-Cart Position",
)
fig.show()

> **Observation:** About 69% of items are added in the first 10 cart positions and about 84% in the first 15. Long carts are rare: only 3,356 orders (about 2.6%) have more than 30 items, and the longest has 80.

---

## Merging Tables to Enrich the Data

Individual tables have IDs, but we want **names**. Merging (joining) lets us combine information across tables.

```python
pd.merge(left_df, right_df, on="shared_column")
```

We'll build an **enriched order-products table** that includes product names, aisle names, and department names.


In [18]:
# Step-by-step merge
op = (
    order_products.merge(
        products, on="product_id"
    )  # add product_name, aisle_id, department_id
    .merge(aisles, on="aisle_id")  # add aisle name
    .merge(departments, on="department_id")  # add department name
)

print(f"Enriched table shape: {op.shape}")
op.head()

Enriched table shape: (1384617, 9)


,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department
0,1,49302,1,1,Bulgarian Yogurt,120,16,yogurt,dairy eggs
1,1,11109,2,1,Organic 4% Milk Fat Whole Milk Cottage Cheese,108,16,other creams cheeses,dairy eggs
2,1,10246,3,0,Organic Celery Hearts,83,4,fresh vegetables,produce
3,1,49683,4,0,Cucumber Kirby,83,4,fresh vegetables,produce
4,1,43633,5,1,Lightly Smoked Sardines in Olive Oil,95,15,canned meat seafood,canned goods


:::{tip}

After merging, always check the shape and a few rows to confirm the join worked as expected. The row count should stay the same if you're doing a many-to-one join (which we are here).

:::

---

## Categorical Analysis (Value Counts & Proportions)

For categorical columns, **value counts** reveal which categories dominate.

:::{tip}

`.value_counts()` returns categories sorted by frequency, with the most common first.

:::

### Top 20 Products


In [19]:
top20 = op["product_name"].value_counts().head(20).reset_index()
top20.columns = ["Product", "Times Ordered"]

fig = px.bar(
    top20,
    x="Times Ordered",
    y="Product",
    orientation="h",
    title="Top 20 Most Ordered Products",
    text_auto=True,
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

### Orders by Department


In [20]:
dept_counts = op["department"].value_counts().reset_index()
dept_counts.columns = ["Department", "Items Ordered"]

fig = px.bar(
    dept_counts,
    x="Items Ordered",
    y="Department",
    orientation="h",
    title="Number of Items Ordered by Department",
    text_auto=True,
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

### Proportion of reordered vs. first-time items


In [21]:
reorder_counts = order_products["reordered"].value_counts().reset_index()
reorder_counts.columns = ["Reordered", "Count"]
reorder_counts["Reordered"] = reorder_counts["Reordered"].map(
    {1: "Reordered", 0: "First time"}
)

fig = px.pie(
    reorder_counts,
    values="Count",
    names="Reordered",
    title="Reordered vs. First-Time Items",
)
fig.show()

---

## Bivariate Analysis (Two Variables Together)

Bivariate analysis explores the **relationship between two variables**. Common techniques:

| Technique         | When to Use                                          |
| ----------------- | ---------------------------------------------------- |
| Grouped bar chart | Compare a metric across categories                   |
| Scatter plot      | Two continuous variables                             |
| Box plot          | Distribution of a numeric variable across categories |

### Reorder ratio by department

> **Question:** Which departments have the highest share of _repeat purchases_?


In [22]:
reorder_dept = (
    op.groupby("department")["reordered"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
reorder_dept.columns = ["Department", "Reorder Ratio"]

fig = px.bar(
    reorder_dept,
    x="Reorder Ratio",
    y="Department",
    orientation="h",
    title="Reorder Ratio by Department (higher = more repeat purchases)",
    text_auto=".2f",
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Insight:** _Dairy eggs_, _produce_, and _beverages_ have the highest reorder ratios (about 0.66 to 0.68). These are routine essentials. _Personal care_ has the lowest (about 0.34), just below _pantry_ and _international_. One explanation: fresh staples run out quickly and get restocked, while items such as shampoo or spices last longer between purchases.

### Average cart position by department


In [23]:
cart_dept = (
    op.groupby("department")["add_to_cart_order"].mean().sort_values().reset_index()
)
cart_dept.columns = ["Department", "Avg Cart Position"]

fig = px.bar(
    cart_dept,
    x="Avg Cart Position",
    y="Department",
    orientation="h",
    title="Average Add-to-Cart Position by Department",
    text_auto=".1f",
    height=600,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

> **Insight:** _Alcohol_ is added to the cart first on average (position about 5.6), followed by _beverages_ and _dairy eggs_. _Babies_ items are added last (about 11.1), after _dry goods pasta_ and _international_. One reading: customers who open the app for a specific purpose (drinks, staples) add those items first and add pantry and specialty items later. Keep in mind that a department's average position also depends on the size of the baskets it appears in: alcohol tends to appear in smaller baskets and babies items in larger ones, so part of this ranking reflects basket size.

### Distribution of basket sizes


In [24]:
basket_sizes = order_products.groupby("order_id").size()
print(basket_sizes.describe().round(1))

# px.box would embed all 131,209 basket sizes in the figure.
# Compute the box-plot statistics here and pass only those numbers to Plotly.
q1, median, q3 = basket_sizes.quantile([0.25, 0.5, 0.75])
iqr = q3 - q1
lower_fence = basket_sizes[basket_sizes >= q1 - 1.5 * iqr].min()
upper_fence = basket_sizes[basket_sizes <= q3 + 1.5 * iqr].max()
n_above = (basket_sizes > upper_fence).sum()
print(f"\nOrders above the upper whisker ({upper_fence} items): {n_above:,}")

fig = go.Figure(
    go.Box(
        q1=[q1],
        median=[median],
        q3=[q3],
        lowerfence=[lower_fence],
        upperfence=[upper_fence],
        mean=[basket_sizes.mean()],
        name="All orders",
    )
)
fig.update_layout(
    title="Box Plot of Items per Order (Basket Size)",
    yaxis_title="Number of Items",
    template="simple_white",
)
fig.show()

count    131209.0
mean         10.6
std           7.9
min           1.0
25%           5.0
50%           9.0
75%          14.0
max          80.0
dtype: float64

Orders above the upper whisker (27 items): 5,222


> **Observation:** The median basket has **9 items** and the middle half of baskets has 5 to 14 items. The mean (about 10.6) sits above the median because the distribution is right-skewed: 5,222 orders have more than 27 items (the upper whisker), and the largest has 80.

---

## Time-Based Patterns

Time-based analysis is extremely common in EDA. We'll look at ordering patterns across days and hours.

### Orders by day of week

:::{caution} Which day is 0?

`order_dow` is stored as a number from 0 to 6. Instacart's data dictionary only says it is "the day of the week the order was placed on" and does not say which number is which day. A common assumption in the community is **0 = Saturday, 1 = Sunday, ..., 6 = Friday**, and this notebook uses it to label the charts. Treat the day names as an assumption, not a fact. The counts per code do not depend on it.

:::

In [25]:
# Assumed mapping: Instacart did not document which code is which day (see the caution above)
dow_map = {
    0: "Saturday",
    1: "Sunday",
    2: "Monday",
    3: "Tuesday",
    4: "Wednesday",
    5: "Thursday",
    6: "Friday",
}
dow_counts = (
    orders["order_dow"]
    .map(dow_map)
    .value_counts()
    .reindex([dow_map[i] for i in range(7)])
    .reset_index()
)
dow_counts.columns = ["Day of Week", "Number of Orders"]

fig = px.bar(
    dow_counts,
    x="Day of Week",
    y="Number of Orders",
    title="Orders by Day of Week",
    text_auto=True,
)
fig.show()

> **Observation:** Codes 0 and 1 (Saturday and Sunday under the assumed mapping) have the most orders, about 600,000 and 590,000. The other five days each have between about 425,000 and 470,000. Under the assumed mapping, grocery ordering leans toward the weekend.

### Orders by hour of day, weekend vs. weekday

There are five weekdays but only two weekend days, so raw counts would always put the weekday line far above the weekend line. To compare the **shape** of the two days, we convert each count into a share of that day type's orders. Under the assumed mapping, codes 0 and 1 are the weekend.

In [26]:
orders_time = orders.copy()
orders_time["day_type"] = (
    orders_time["order_dow"].isin([0, 1]).map({True: "Weekend", False: "Weekday"})
)

hour_daytype = (
    orders_time.groupby(["order_hour_of_day", "day_type"])
    .size()
    .reset_index(name="count")
)
# Share of each day type's orders that fall in each hour
hour_daytype["share"] = hour_daytype["count"] / hour_daytype.groupby("day_type")[
    "count"
].transform("sum")

fig = px.line(
    hour_daytype,
    x="order_hour_of_day",
    y="share",
    color="day_type",
    title="Share of Orders by Hour: Weekend vs Weekday",
    labels={
        "order_hour_of_day": "Hour of Day",
        "share": "Share of the Day Type's Orders",
        "day_type": "Day Type",
    },
    markers=True,
)
fig.update_yaxes(tickformat=".0%")
fig.show()

> **Observation:** Once each line is scaled to its own total, the two shapes are very similar. Weekday orders start slightly earlier (a larger share at 6 to 7 AM) and run slightly later into the evening (5 to 8 PM). Weekend orders are a little more concentrated between 9 AM and 4 PM.

### Heatmap: Day of week × Hour of day


In [27]:
heatmap_data = (
    orders_time.assign(day_name=orders_time["order_dow"].map(dow_map))
    .groupby(["day_name", "order_hour_of_day"])
    .size()
    .reset_index(name="count")
    .pivot(index="day_name", columns="order_hour_of_day", values="count")
    .reindex([dow_map[i] for i in range(7)])
)

fig = px.imshow(
    heatmap_data,
    title="Order Heatmap: Day of Week × Hour of Day",
    labels=dict(x="Hour of Day", y="Day of Week", color="Orders"),
    aspect="auto",
    color_continuous_scale="Blues",
)
fig.show()

> **Insight:** The two hottest blocks are code 1 (Sunday under the assumed mapping) in the morning, peaking at 10 AM, and code 0 (Saturday) in the early afternoon, peaking around 1 to 3 PM. On the other five days, orders are spread fairly evenly from about 9 AM to 4 PM.

---

## Correlation

### Numeric correlation

Correlation measures the **linear relationship** between two numeric variables (−1 to +1).

In [28]:
# Compute correlations for the orders table
orders_numeric = orders[
    ["order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"]
].dropna()
corr = orders_numeric.corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    title="Correlation Matrix: Orders Table",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="equal",
)
fig.show()

> **Reading the matrix:** Most values are close to 0, so most pairs of columns have little linear relationship. The exception is `order_number` and `days_since_prior_order` (about −0.36): orders that come later in a customer's sequence tend to follow shorter gaps. Customers who shop often build up many orders, and they also have short gaps between them.

### Reorder rate versus volume by aisle

Do the aisles that sell the most also get reordered the most? We take the 15 aisles with the most items ordered, compute each aisle's reorder rate and item count with `groupby`, and plot one against the other.

In [29]:
# Reorder rate by aisle (top 15 aisles by volume)
top15_aisles = op["aisle"].value_counts().head(15).index

aisle_reorder = (
    op[op["aisle"].isin(top15_aisles)]
    .groupby("aisle")["reordered"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
    .reset_index()
)
aisle_reorder.columns = ["Aisle", "Reorder Rate", "Total Items"]

fig = px.scatter(
    aisle_reorder,
    x="Total Items",
    y="Reorder Rate",
    text="Aisle",
    title="Reorder Rate vs Volume for Top 15 Aisles",
    labels={"Total Items": "Number of Items Ordered", "Reorder Rate": "Reorder Rate"},
)
fig.update_traces(textposition="top center")
fig.show()

> **Insight:** _Fresh fruits_ combines the highest volume with one of the highest reorder rates (about 0.74). _Fresh vegetables_ sells just as much but is reordered less often (about 0.61), which fits Instacart's own note that vegetables are bought more intermittently for recipes. _Milk_ has the highest reorder rate of the 15 (about 0.79) at much lower volume, and _ice cream ice_ has the lowest (about 0.48). Volume and reorder rate are different signals: a high-volume aisle is not automatically a high-reorder aisle.

---

## Summary

- **Tables.** `orders` has 3.4 million orders from 206,209 users, but `order_products` covers only 131,209 of them: the train split, which holds one order per user (that user's most recent order).
- **Missing values.** Only `days_since_prior_order` has missing values, one per user, because a first order has no prior order.
- **Timing.** About two thirds of orders are placed between 9 AM and 5 PM. Codes 0 and 1 of `order_dow` are the two busiest days; calling them Saturday and Sunday is an assumption.
- **Cadence.** Gaps between orders spike at 7 days and at the 30-day cap.
- **Baskets.** The median basket has 9 items and the distribution is right-skewed.
- **Products.** Produce and dairy eggs lead on volume, and about 60% of items are reorders. Dairy eggs, produce, and beverages have the highest reorder ratios; personal care has the lowest.
- **Charts.** For columns with millions of rows, count first and plot the counts. The figures stay small and look the same.